In [8]:
from IPython.display import display, HTML

display(HTML('<style>body {margin: 0 !important; padding: 0 !important;}</style>'))
display(HTML('<style>.jp-WindowedPanel-outer {padding: 0 !important; margin: 0 !important;}</style>')) 
display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>.container { height:100% !important; }</style>"))

import dash
from dash import html, dcc, Input, Output, State, ALL, ctx
import json
import random

app = dash.Dash(__name__)

# Custom CSS to ensure full height
app.index_string = '''
<!DOCTYPE html>
<html>
    <head>
        {%metas%}
        <title>{%title%}</title>
        {%favicon%}
        {%css%}
        <style>
            html, body, #react-entry-point, ._dash-loading {
                height: 100%;
                margin: 0;
                padding: 0;
                overflow-x: hidden;
            }
        </style>
    </head>
    <body>
        {%app_entry%}
        <footer>
            {%config%}
            {%scripts%}
            {%renderer%}
        </footer>
    </body>
</html>
'''

app.layout = html.Div([
    html.H1("Dice Pool Roller"),

    # Store for dice configuration
    dcc.Store(id='dice-store', data=[]),

    # Controls
    html.Div([
        html.Label('Sides:', style={'marginRight': '5px'}),
        dcc.Input(id='new-die-sides', type='number', min=1, max=100, value=6, style={'width': '60px', 'marginRight': '10px'}),
        html.Button('Add Die', id='add-die-btn', n_clicks=0),
        html.Button('Roll Pool', id='roll-btn', n_clicks=0, 
                   style={'marginLeft': '20px', 'backgroundColor': '#4CAF50', 'color': 'white'})
    ], style={'marginBottom': '20px', 'display': 'flex', 'alignItems': 'center'}),

    # Dice container - vertical stack, grows to fill space
    html.Div(id='dice-container', 
             style={
                 'display': 'flex', 
                 'flexDirection': 'column', 
                 'gap': '10px',
                 'minHeight': '0',
                 'flex': '1'
             }),

    # Results display - stays at bottom
    html.Div(id='roll-results', style={
        'marginTop': 'auto', 
        'paddingTop': '30px',
        'fontSize': '18px'
    })

], style={
    'display': 'flex',
    'flexDirection': 'column',
    'minHeight': '100vh',
    'padding': '20px',
    'boxSizing': 'border-box'
})


# Single callback to handle ALL modifications to dice store
@app.callback(
    Output('dice-store', 'data'),
    Input('add-die-btn', 'n_clicks'),
    Input({'type': 'remove-die-btn', 'index': ALL}, 'n_clicks'),
    Input({'type': 'sides-input', 'index': ALL}, 'value'),
    Input({'type': 'side-value-input', 'index': ALL, 'side': ALL}, 'value'),
    State('dice-store', 'data'),
    State('new-die-sides', 'value'),
    prevent_initial_call=True
)
def update_dice_store(add_clicks, remove_clicks, sides_values, side_values, current_data, new_die_sides):
    if not ctx.triggered:
        return current_data

    triggered_id = ctx.triggered_id

    # Handle Add Die
    if triggered_id == 'add-die-btn':
        sides = new_die_sides if new_die_sides and new_die_sides > 0 else 6
        new_die = {
            'sides': sides,
            'values': list(range(1, sides + 1))
        }
        current_data.append(new_die)
        return current_data

    # Handle Remove Die
    if isinstance(triggered_id, dict) and triggered_id.get('type') == 'remove-die-btn':
        index_to_remove = triggered_id['index']
        if 0 <= index_to_remove < len(current_data):
            current_data.pop(index_to_remove)
        return current_data

    # Handle Sides Count Change (within card)
    if isinstance(triggered_id, dict) and triggered_id.get('type') == 'sides-input':
        die_index = triggered_id['index']
        new_sides = ctx.triggered[0]['value']

        if new_sides is None or new_sides < 1 or die_index >= len(current_data):
            return current_data

        old_values = current_data[die_index]['values']

        if len(old_values) < new_sides:
            for i in range(len(old_values), new_sides):
                old_values.append(i + 1)
        else:
            old_values = old_values[:new_sides]

        current_data[die_index]['sides'] = new_sides
        current_data[die_index]['values'] = old_values
        return current_data

    # Handle Individual Side Value Change
    if isinstance(triggered_id, dict) and triggered_id.get('type') == 'side-value-input':
        die_index = triggered_id['index']
        side_index = triggered_id['side']
        new_value = ctx.triggered[0]['value']

        if (0 <= die_index < len(current_data) and 
            0 <= side_index < len(current_data[die_index]['values']) and 
            new_value is not None):
            current_data[die_index]['values'][side_index] = new_value
        return current_data

    return current_data


# Render dice UI
@app.callback(
    Output('dice-container', 'children'),
    Input('dice-store', 'data')
)
def render_dice(dice_data):
    if not dice_data:
        return html.Em("No dice in pool. Click 'Add Die' to start.")

    dice_elements = []

    for i, die in enumerate(dice_data):
        # Horizontal layout for side value inputs
        side_inputs = []
        for j, value in enumerate(die['values']):
            side_inputs.append(
                dcc.Input(
                    type='number',
                    value=value,
                    id={'type': 'side-value-input', 'index': i, 'side': j},
                    style={
                        'width': '50px', 
                        'textAlign': 'center',
                        'marginRight': '8px',
                        'marginBottom': '5px'
                    }
                )
            )

        # Die card
        die_card = html.Div([
            # Header row
            html.Div([
                html.Strong(f'Die #{i+1}', style={'minWidth': '60px'}),

                html.Div([
                    html.Label('Sides:', style={'marginRight': '5px', 'fontSize': '14px'}),
                    dcc.Input(
                        type='number',
                        min=1,
                        max=100,
                        value=die['sides'],
                        id={'type': 'sides-input', 'index': i},
                        style={'width': '60px'}
                    )
                ], style={'display': 'flex', 'alignItems': 'center', 'marginLeft': '20px'}),

                html.Button('×', id={'type': 'remove-die-btn', 'index': i}, 
                          style={'marginLeft': 'auto', 'color': 'red', 'cursor': 'pointer', 'fontSize': '16px'})

            ], style={
                'display': 'flex',
                'alignItems': 'center',
                'borderBottom': '1px solid #eee',
                'paddingBottom': '8px',
                'marginBottom': '8px'
            }),

            # Side values
            html.Div(side_inputs, style={
                'display': 'flex',
                'flexWrap': 'wrap',
                'overflowX': 'auto',
                'paddingTop': '5px'
            })

        ], style={
            'border': '1px solid #ddd',
            'borderRadius': '5px',
            'padding': '12px',
            'backgroundColor': '#f9f9f9',
            'flexShrink': '0'
        })

        dice_elements.append(die_card)

    return dice_elements


# Roll the dice pool
@app.callback(
    Output('roll-results', 'children'),
    Input('roll-btn', 'n_clicks'),
    State('dice-store', 'data'),
    prevent_initial_call=True
)
def roll_dice(n_clicks, dice_data):
    if not dice_data:
        return html.Div("No dice to roll!", style={'color': 'red'})

    results = []
    details = []

    for i, die in enumerate(dice_data):
        if die['values']:
            roll_result = random.choice(die['values'])
            results.append(roll_result)
            details.append(f"Die #{i+1}: {roll_result}")

    total = sum(results)

    return html.Div([
        html.Hr(),
        html.Strong("Roll Results"),
        html.Div([html.Div(d) for d in details], style={'margin': '10px 0', 'fontFamily': 'monospace'}),
        html.Div(f"Individual Results: {results}", style={'marginTop': '10px'}),
        html.Div(f"Total Sum: {total}", style={'marginTop': '5px', 'fontSize': '24px', 'fontWeight': 'bold', 'color': '#4CAF50'})
    ])


if __name__ == '__main__':
    app.run() 
